In [1]:
import torch
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
import pyarrow.parquet as pq
from torch.utils.data import DataLoader, IterableDataset
import math
import random
from utils.models import *
from utils.train_test import *
from utils.dna_dataset_2 import DaskCuDFIntegerDataset, ParquetCPUDataset
import fsspec
torch.set_float32_matmul_precision('high')


DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
NUM_CLASSES = 13683
MAX_LEN = 512
NUM_FEATURES = 4
BATCH_SIZE = 2048 # Pode ser maior no teste (apenas inferência)
base_type_data = "species"
# Caminhos dos seus Melhores Modelos (Atualize estes caminhos!)
PATH_RESNET = f"../cnn_kraken_{base_type_data}/checkpoints/best_resnet_classifier__fast_phase4.pt" # Exemplo: seu melhor checkpoint da ResNet
PATH_CONVNEXT_zero = f"./checkpoint_{base_type_data}_conv_zero/best_convnext_phase3_restarts_v2.pt" # Seu atual campeão
PATH_CONVNEXT_transfer =  f"./checkpoint_transfer_{base_type_data}_conv/best_convnext_classifier_fast_phase4_2_restarts.pt"
PATH_CONVNEXT_fine =  f"./checkpoint_transfer_{base_type_data}_conv/finetune_noise_7.pt"
PATH_RESNET_fine = f"./checkpoint_finetune_resnet/finetune_noise_14.pt"

/home/mlima/workspace/envs/rapids-new/lib/python3.12/site-packages/torch/__init__.py:1624: UserWarning: Please use the new API settings to control TF32 behavior, such as torch.backends.cudnn.conv.fp32_precision = 'tf32' or torch.backends.cuda.matmul.fp32_precision = 'ieee'. Old settings, e.g, torch.backends.cuda.matmul.allow_tf32 = True, torch.backends.cudnn.allow_tf32 = True, allowTF32CuDNN() and allowTF32CuBLAS() will be deprecated after Pytorch 2.9. Please see https://pytorch.org/docs/main/notes/cuda.html#tensorfloat-32-tf32-on-ampere-and-later-devices (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:80.)
  _C._set_float32_matmul_precision(precision)


In [2]:
# --- 2. Dataset de Teste (Reutilizando sua classe CPU otimizada) ---
# Copie a classe ParquetCPUDataset aqui se ela não estiver no escopo
# (Assumindo que ela já foi definida nas células anteriores)

fs = fsspec.filesystem('file')
test_file_paths = fs.glob(os.path.join(f"./processed_kraken_{base_type_data}_splited_parquet/test", '*.parquet'))

NUCLEOTIDES = ['A', 'C', 'G', 'T']
PADDED_COLUMN = 'sequence_padded'
LABEL_COLUMN = 'label'
BATCH_SIZE = 4096
MAX_SEQUENCE_LENGTH = 512
test_dataset = ParquetCPUDataset(
    test_file_paths, PADDED_COLUMN, LABEL_COLUMN,
    MAX_SEQUENCE_LENGTH, NUCLEOTIDES, BATCH_SIZE
)

test_loader = DataLoader(
    test_dataset, num_workers=8, pin_memory=True, prefetch_factor=2, batch_size=None
)

In [3]:
from utils.dna_dataset_2 import DaskCuDFIntegerDataset

# Parâmetros de configuração
NUM_CLASSES = 13683
#NUM_CLASSES = 2702
MAX_SEQUENCE_LENGTH = 512
NUM_FEATURES = 4
BATCH_SIZE = 8192
#BATCH_SIZE = 496
NUM_WORKERS = 0
NUCLEOTIDES = ['A', 'C', 'G', 'T']
PADDED_COLUMN = 'sequence_padded'
LABEL_COLUMN = 'label'
PREFETCH_FACTOR = None
base_type_data = "species"
#base_type_data = "genus"
# Get Parquet file paths for the splits
fs = fsspec.filesystem('file')
base_output_directory = f"./processed_kraken_{base_type_data}_splited_parquet"
#base_output_directory = "./synthetic_splited_dataset"
train_dir = os.path.join(base_output_directory, "train")
test_dir = os.path.join(base_output_directory, "test")
val_dir = os.path.join(base_output_directory, "validation")
train_file_paths = fs.glob(os.path.join(train_dir, '*.parquet'))
test_file_paths = fs.glob(os.path.join(test_dir, '*.parquet'))
val_file_paths = fs.glob(os.path.join(val_dir, '*.parquet'))
print(f"Found {len(train_file_paths)} train files, {len(test_file_paths)} test files, {len(val_file_paths)} validation files.")


print(f"Configurando DataLoader com {NUM_WORKERS} workers e prefetch de {PREFETCH_FACTOR} lotes.")


test_dataset = DaskCuDFIntegerDataset(
    test_file_paths, PADDED_COLUMN, LABEL_COLUMN,
    MAX_SEQUENCE_LENGTH, NUCLEOTIDES, BATCH_SIZE
)



test_loader = DataLoader(
    test_dataset,
    num_workers=NUM_WORKERS,
    pin_memory=False,
    prefetch_factor=PREFETCH_FACTOR,
    batch_size=None 
)

print("PyTorch DataLoaders criados.")

Found 256 train files, 256 test files, 256 validation files.
Configurando DataLoader com 0 workers e prefetch de None lotes.
PyTorch DataLoaders criados.


In [4]:
from utils.models import *


In [5]:
models = [
    {
        "model_name":"ConvNext finetuned",
        "path":PATH_CONVNEXT_fine,
        "type":"CONVNEXT_FINE"
        
    },
    {
        "model_name":"ConvNext from scratch",
        "path":PATH_CONVNEXT_zero,
        "type":"CONVNEXT"
    },
    {
        "model_name":"ConvNext transfer from genus",
        "path":PATH_CONVNEXT_transfer,
        "type":"CONVNEXT"
    },
    {
        "model_name":"ResNet50",
        "path":PATH_RESNET,
        "type":"RESNET"
    },
    {
        "model_name":"ResNet50_finetuned",
        "path":PATH_RESNET_fine,
        "type":"RESNET_FINE"
    },
]

In [6]:
def load_model(model_dict):
    # 1. Instanciação do Modelo Base
    if "CONVNEXT" in model_dict["type"]:
        dropout = 0.7 if model_dict["type"] == "CONVNEXT_FINE" else 0.5
        model = convnext_tiny_1d(MAX_SEQUENCE_LENGTH, NUM_FEATURES, NUM_CLASSES, debug=False, dropout_rate=dropout)
    elif "RESNET" in model_dict["type"]:
        model = resnet50_1d(MAX_SEQUENCE_LENGTH, NUM_FEATURES, NUM_CLASSES, debug=False, dropout_rate=0.5)
    else:
        raise ValueError(f"Tipo de modelo desconhecido: {model_dict['type']}")

    # 2. Carregamento dos Pesos no modelo 'bruto'
    # Usar map_location garante que os pesos caiam no lugar certo desde o início
    try:
        print(f"Carregando pesos de: {model_dict['path']}")
        checkpoint = torch.load(model_dict["path"], map_location=DEVICE)
        
        # Extrai o dicionário de estados (suporta formatos simples ou com metadados de treino)
        state_dict = checkpoint["model_state_dict"] if isinstance(checkpoint, dict) and "model_state_dict" in checkpoint else checkpoint
        
        # Limpa o prefixo '_orig_mod.' caso o checkpoint tenha sido salvo de um modelo já compilado
        new_state_dict = {k.replace('_orig_mod.', ''): v for k, v in state_dict.items()}
        
        model.load_state_dict(new_state_dict)
        print(f"Sucesso! Pesos carregados para {model_dict['model_name']}.")
    except Exception as e:
        print(f"ATENÇÃO: Falha ao carregar checkpoint: {e}")
        print("O modelo seguirá com inicialização aleatória.")

    # 3. Preparação para Inferência
    model.to(DEVICE)
    model.eval() # Essencial para congelar BatchNorm e Dropout antes da compilação

    # 4. Compilação (O ÚLTIMO PASSO)
    # Compilar apenas após o modelo estar com os pesos finais e no device correto
    print(f"Compilando modelo {model_dict['model_name']}...")
    model = torch.compile(model)
    
    return model

In [7]:
import time
import torch
import torch.nn as nn
import numpy as np

# =====================================================================
# 1. Benchmark Profiling Engine
# =====================================================================

def profile_model(model, input_size=(32, 4, 512), device='cpu', num_warmup=15, num_runs=100):
    """
    Profiles a PyTorch model to compute:
    - Parameter counts (Total and Trainable)
    - Model footprint in memory (MB)
    - Mean latency and standard deviation (ms per batch)
    - Inference throughput (sequences processed per second)
    """
    model = model.to(device)
    model.eval()

    # Create dummy input batch reflecting the spatial genomic tensor shape
    x = torch.randint(0, 4, input_size, dtype=torch.long, device=device)

    # 1. Calculate Parameter Statistics
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

    # 2. Calculate Memory Footprint (Parameters + Buffers like BatchNorm running states)
    param_size = sum(p.numel() * p.element_size() for p in model.parameters())
    buffer_size = sum(b.numel() * b.element_size() for b in model.buffers())
    total_size_mb = (param_size + buffer_size) / (1024 ** 2)

    # 3. Warmup Passes
    # Crucial to eliminate PyTorch's initial graph allocation and CUDA JIT/driver overhead
    with torch.no_grad():
        for _ in range(num_warmup):
            _ = model(x)
        if device == 'cuda':
            torch.cuda.synchronize()

    # 4. Latency Benchmarking
    latencies = []
    with torch.no_grad():
        for _ in range(num_runs):
            if device == 'cuda':
                torch.cuda.synchronize() # Force GPU to finish execution before stopping clock
            
            start_time = time.perf_counter()
            _ = model(x)
            
            if device == 'cuda':
                torch.cuda.synchronize() # Wait for operations to finish
            end_time = time.perf_counter()
            
            latencies.append((end_time - start_time) * 1000) # Convert to milliseconds

    avg_latency = np.mean(latencies)
    std_latency = np.std(latencies)
    
    # 5. Throughput Computation
    batch_size = input_size[0]
    throughput = (batch_size / (avg_latency / 1000.0))

    return {
        'total_params': total_params,
        'trainable_params': trainable_params,
        'size_mb': total_size_mb,
        'avg_latency_ms': avg_latency,
        'std_latency_ms': std_latency,
        'throughput_seq_sec': throughput
    }




In [12]:
    # Automatically switch to CUDA GPU if available (recommended for exact RTX 6000 measurements)
    target_device = 'cuda' if torch.cuda.is_available() else 'cpu'
    batch_size = 32 # Recommended batch size representing actual inference workload
    
    print(f"=== Genomic 1D-CNN Model Profiler ===")
    print(f"Running on Device: {target_device.upper()}")
    print(f"Configuration: Batch Size = {batch_size}, Input Dimensions = (4, 512)\n")
    results = {}
    for model_dict in models:
        model = load_model(model_dict)


        name = model_dict['model_name']
        
        print(f"Profiling {name}...")
        results[name] = profile_model(model, input_size=(batch_size, 512), device=target_device)



=== Genomic 1D-CNN Model Profiler ===
Running on Device: CUDA
Configuration: Batch Size = 32, Input Dimensions = (4, 512)

Carregando pesos de: ./checkpoint_transfer_species_conv/finetune_noise_7.pt
Sucesso! Pesos carregados para ConvNext finetuned.
Compilando modelo ConvNext finetuned...
Profiling ConvNext finetuned...
Carregando pesos de: ./checkpoint_species_conv_zero/best_convnext_phase3_restarts_v2.pt
Sucesso! Pesos carregados para ConvNext from scratch.
Compilando modelo ConvNext from scratch...
Profiling ConvNext from scratch...
Carregando pesos de: ./checkpoint_transfer_species_conv/best_convnext_classifier_fast_phase4_2_restarts.pt
Sucesso! Pesos carregados para ConvNext transfer from genus.
Compilando modelo ConvNext transfer from genus...
Profiling ConvNext transfer from genus...
Carregando pesos de: ../cnn_kraken_species/checkpoints/best_resnet_classifier__fast_phase4.pt
Sucesso! Pesos carregados para ResNet50.
Compilando modelo ResNet50...
Profiling ResNet50...
Carregando 

In [13]:
    
# Print comparative results
print("\n" + "="*80)
print(f"{'Model Name':<20} | {'Total Params':<15} | {'Size (MB)':<10} | {'Latency (ms)':<15} | {'Throughput':<15}")
print("="*80)
for name, m in results.items():
    print(f"{name:<20} | {m['total_params']::<15} | {m['size_mb']:.2f} MB   | {m['avg_latency_ms']:.2f} ± {m['std_latency_ms']:.2f} | {m['throughput_seq_sec']:.1f} seq/s")
print("="*80)


Model Name           | Total Params    | Size (MB)  | Latency (ms)    | Throughput     
ConvNext finetuned   | 164013811:::::: | 625.66 MB   | 4.06 ± 0.09 | 7889.8 seq/s
ConvNext from scratch | 164013811:::::: | 625.66 MB   | 4.06 ± 0.07 | 7877.0 seq/s
ConvNext transfer from genus | 164013811:::::: | 625.66 MB   | 4.08 ± 0.05 | 7844.1 seq/s
ResNet50             | 43992051::::::: | 168.02 MB   | 3.30 ± 0.04 | 9686.3 seq/s
ResNet50_finetuned   | 43992051::::::: | 168.02 MB   | 3.31 ± 0.07 | 9666.2 seq/s
